# U2T01 — the last SQuAD run, on a Kaggle GPU

Nineteen of the twenty runs are already done and ship inside this project's `results/`.
Only **full fine-tuning on SQuAD v1.1** is missing, and it is the heaviest one — which is why
it runs here instead of on a laptop.

This notebook runs the repository's own modules; nothing is reimplemented, so the
`results/qa/*.json` it produces drops straight into the report beside the local runs.

**Before running:** in the right-hand panel set *Accelerator* to **GPU P100** (or T4 x2) and
*Internet* to **On**. Both need a phone-verified account; without internet the datasets and
the `bert-base-uncased` weights cannot be downloaded.


## 1. Unpack the project


In [ ]:
import glob, os, pathlib, shutil

WORK = pathlib.Path('/kaggle/working/u2t01')
zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
assert zips, 'Attach u2t01_portable.zip through + Add Input -> Upload'

!mkdir -p {WORK} && unzip -q -o "{zips[0]}" -d {WORK}
if not (WORK / 'src').is_dir():          # zip may hold a single top-level folder
    WORK = next(p for p in WORK.iterdir() if (p / 'src').is_dir())
os.chdir(WORK)

print('working directory:', pathlib.Path.cwd())
print('results already present:', len(glob.glob('results/*/*.json')), 'runs')


## 2. Dependencies

Kaggle already ships torch, numpy, pandas and matplotlib built against its CUDA image.
Replacing them would break that image, so only the libraries this project actually drives
are installed, at the versions the rest of the results were produced with.


In [ ]:
# Not quiet, and not piped through tail: a pip failure here is the difference between
# a 20-minute run and a crash, and it must be visible.
!pip install transformers==4.57.6 datasets==5.0.1 'accelerate>=1.0' seqeval evaluate

# Verify in a SUBPROCESS - the notebook kernel may still hold a previously imported version,
# and the training scripts run as subprocesses, so that is the version that actually matters.
!python -c "import torch, transformers, datasets; \
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'); \
print('bfloat16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else '-'); \
print('transformers', transformers.__version__, '| datasets', datasets.__version__); \
assert torch.cuda.is_available(), 'Turn the GPU on in the runtime settings'; \
assert transformers.__version__.startswith('4.57'), \
  f'transformers {transformers.__version__} is installed, not 4.57.x - rerun the pip cell and check its output'"


The precision is picked from what the card supports: bfloat16 on Ampere and newer, float16
on a P100 or T4. Asking a pre-Ampere card for bfloat16 makes `TrainingArguments` refuse to start.


## 3. Check the data before training

A silent subword/label misalignment produces a perfectly healthy-looking loss curve and a
model that learned nothing. Thirteen checks, all of which should pass.


In [ ]:
!python scripts/verify_data.py 2>&1 | grep -E '^(PASS|FAIL|\[)'


## 4. Run it

The grid is resumable, so the two SQuAD runs that already finished are skipped and only full
fine-tuning actually trains. Expect roughly 15–25 minutes on a P100.


In [ ]:
!python -u scripts/run_experiments.py --task qa 2>&1 | grep -vE '^ *[0-9]+%\|'


## 5. What came out


In [ ]:
import sys; sys.path.insert(0, '.')
from src.report_data import rows
for r in rows('qa'):
    if r['headline'] is not None:
        print(f"{r['method']:34s} F1 {r['headline']:6.2f}   EM {r['secondary']:6.2f}   "
              f"trainable={r['trainable']:>11,}   {r['minutes']:5.1f} min")


In [ ]:
# Confirm the checkpoint on disk reproduces the metric that was just reported.
!python scripts/verify_models.py --all --task qa 2>&1 | grep -E '^(OK|FAIL|\[)'


## 6. Bring the results back

`results/` is what the report reads — download it and hand it over. It is a few hundred
kilobytes.


In [ ]:
!zip -qr /kaggle/working/u2t01_qa_results.zip results docs
!ls -lh /kaggle/working/u2t01_qa_results.zip
print('Download it from the Output panel on the right.')


### The checkpoint

The model itself is about 420 MB. Two ways to get it where it needs to go:

**a) Push it to the Hub from here** — no large download at all. Needs a write token from
<https://huggingface.co/settings/tokens>.

**b) Download it** from the Output panel (`models/qa/full_ft_linear_bert-base-uncased/`) and
publish from the laptop later.


In [ ]:
# Option (a). Skip this cell if you would rather download the checkpoint.
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
!python scripts/push_to_hub.py --user joseeangel --tasks qa
